In [0]:
%pip install langchain langchain-core databricks-langchain langgraph --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
mlflow.langchain.autolog()

In [0]:
llm_endpoint_name = "databricks-gpt-oss-120b"
vs_index_name='dmi_genomics_qa_team_consenting_practice.bg_genai.docs_chunked_index'

In [0]:
from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langgraph.checkpoint.memory import InMemorySaver


def build_agent(llm_endpoint:str, index_name: str, num_results: int = 3):
    model = ChatDatabricks(
        endpoint=llm_endpoint,
        max_tokens=500,
    )

    vs_tool = VectorSearchRetrieverTool(
        name="spark_knowledge_search",
        index_name=index_name,
        description="Search spark, databricks knowledge base for relevant information",
        num_results=num_results,
    )

    # Optional: use a in memory saver to save the agent's state
    checkpointer = InMemorySaver()

    system_prompt = """You are the Spark Knowledge Assistant (SKA). Respond in a clear, professional, and factual tone appropriate for engineers and technical staff. Use only verified information from spark's internal documents, and include source references when available. If the answer cannot be found, clearly state that and suggest related sections or next steps. Do not speculate, make assumptions, or provide information outside the provided context."""

    agent = create_agent(
        model=model, 
        tools=[vs_tool], 
        system_prompt=system_prompt,
        checkpointer=checkpointer,
        )
    return agent

# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "databricks-demo-4"}}

# Quick smoke test
agent = build_agent(llm_endpoint_name, vs_index_name, 3)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is spark?"}]},
    config=config
)
print(response['messages'][-1].content)

Tool name dmi_genomics_qa_team_consenting_practice__bg_genai__docs_chunked_index is too long, truncating to 64 characters nomics_qa_team_consenting_practice__bg_genai__docs_chunked_index.


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
**Apache Spark** is an open‑source, distributed data‑processing engine that coordinates the execution of tasks on large data sets across a cluster of computers.  

- It provides a framework for **managing and scheduling work** on many machines so that the combined resources of the cluster can be used as a single, powerful compute engine.  
- Spark runs on top of a **cluster manager** (e.g., Spark’s own Standalone manager, YARN, or Mesos) that allocates resources to a Spark application.  
- Applications are submitted with tools such as **`spark‑submit`**, and can be written in any of Spark’s supported languages (Scala, Java, Python, R).  

In short, Spark is the “glue” that lets you write big‑data programs once and have them executed efficiently on a distributed cluster, turning i

Trace(trace_id=tr-ebd60cab6a3e6bd3ee290c0ca50a13a9)

In [0]:
import yaml

def create_config(llm_endpoint_name: str, index_name: str, num_results: int = 3):
    """Create a minimal YAML config for the agent."""
    config = {
        "llm_endpoint_name": llm_endpoint_name,
        "vector_search": {
            "index_name": index_name,
            "num_results": num_results
        }
    }
    return config


# Create config file
llm_endpoint_name = "databricks-gpt-oss-120b"

agent_config = create_config(llm_endpoint_name, vs_index_name)

# Write YAML file (for agent.py to read later)
with open("agent-config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(agent_config, f, sort_keys=False)

print("✅ Config file written: agent-conf.yaml")
print(yaml.safe_dump(agent_config, sort_keys=False))

✅ Config file written: agent-conf.yaml
llm_endpoint_name: databricks-gpt-oss-120b
vector_search:
  index_name: dmi_genomics_qa_team_consenting_practice.bg_genai.docs_chunked_index
  num_results: 3



In [0]:
%%writefile agent.py
import os
from uuid import uuid4
from typing import Any, Dict, List

import yaml
import mlflow
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse

from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langgraph.checkpoint.memory import InMemorySaver

# Load agent configuration from YAML file
def _load_config(path: str = "agent-config.yaml") -> Dict[str, Any]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Config file not found at '{path}'")
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    llm_endpoint = cfg.get("llm_endpoint_name")
    vs = cfg.get("vector_search", {}) or {}
    index_name = vs.get("index_name")
    num_results = int(vs.get("num_results", 3))
    if not llm_endpoint or not index_name:
        raise ValueError("Missing 'llm_endpoint_name' or 'vector_search.index_name' in agent-config.yaml")
    return {
        "llm_endpoint_name": llm_endpoint,
        "vs_index_name": index_name,
        "vs_num_results": num_results,
    }

# Build LangChain agent with LLM and vector search tool
def build_agent(llm_endpoint: str, index_name: str, num_results: int = 3):
    model = ChatDatabricks(endpoint=llm_endpoint, max_tokens=500)
    vs_tool = VectorSearchRetrieverTool(
        name="spark_knowledge_search",
        index_name=index_name,
        description="Search Spark knowledge base for relevant information",
        num_results=num_results,
    )
    checkpointer = InMemorySaver()
    system_prompt = (
        "You are the Spark Knowledge Assistant (SKA). Respond in a clear, professional, and factual tone "
        "appropriate for engineers and technical staff. Use only verified information from Spark's internal "
        "documents, and include source references when available. If the answer cannot be found, clearly state "
        "that and suggest related sections or next steps. Do not speculate, make assumptions, or provide "
        "information outside the provided context."
    )
    agent = create_agent(
        model=model,
        tools=[vs_tool],
        system_prompt=system_prompt,
        checkpointer=checkpointer,
    )
    return agent

# Extract last user message from conversation
def _last_user_text(messages: List[Dict[str, Any]]) -> str:
    user_msgs = [m for m in messages if (m.get("role") == "user")]
    return str(user_msgs[-1].get("content", "")) if user_msgs else str(messages[-1].get("content", ""))

# MLflow ResponsesAgent implementation for LangChain agent
class LangChainResponsesAgent(ResponsesAgent):
    def __init__(self):
        cfg = _load_config()
        self._cfg = cfg
        self._agent = build_agent(
            llm_endpoint=cfg["llm_endpoint_name"],
            index_name=cfg["vs_index_name"],
            num_results=cfg["vs_num_results"],
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        msgs = [m.model_dump() for m in request.input]  # [{'role': 'user'|'assistant', 'content': '...'}, ...]
        _ = _last_user_text(msgs) if msgs else ""

        # Generate a unique thread ID for each prediction
        thread_id = f"oka-{uuid4()}"

        result = self._agent.invoke(
            {"messages": msgs},
            config={"configurable": {"thread_id": thread_id}},
        )
        # Extract agent response text
        try:
            text = result["messages"][-1].content
        except Exception:
            text = str(result)

        return ResponsesAgentResponse(
            output=[self.create_text_output_item(text, str(uuid4()))],
            custom_outputs=request.custom_inputs,
        )

# Set the model for mlflow. This is needed when using agent-as-code approach
AGENT = LangChainResponsesAgent()
mlflow.models.set_model(AGENT)

Writing agent.py


In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
from agent import AGENT as agent

mlflow.langchain.autolog()

response  = agent.predict(
    {"input": [{"role": "user", "content": "What is Spark?"}]}
)


Tool name dmi_genomics_qa_team_consenting_practice__bg_genai__docs_chunked_index is too long, truncating to 64 characters nomics_qa_team_consenting_practice__bg_genai__docs_chunked_index.


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Trace(trace_id=tr-60da82be94f4db60cee44265f5d1b80f)

In [0]:
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint

# Redefine variables after Python restart
vs_index_name = "dmi_genomics_qa_team_consenting_practice.bg_genai.docs_chunked_index"
llm_endpoint_name = "databricks-gpt-oss-120b"

# Define the resources that the agent depends on
resources = [
    DatabricksVectorSearchIndex(index_name=vs_index_name),
    DatabricksServingEndpoint(endpoint_name=llm_endpoint_name)
]

print("Resources defined:")
for resource in resources:
    print(f"  - {resource}")

Resources defined:
  - <mlflow.models.resources.DatabricksVectorSearchIndex object at 0x7f5045924c80>
  - <mlflow.models.resources.DatabricksServingEndpoint object at 0x7f5043669d30>


In [0]:
import mlflow
from importlib.metadata import version as get_version

# Define model name and tags
model_name = "spark_knowledge_assistant"
tags_to_register = {
    "model_type": "retrieval_agent",
    "framework": "langchain",
    "use_case": "orion_knowledge_base"
}

# Create an input example for the model signature
input_example = {
    "input": [
        {"role": "user", "content": "What is Spark?"}
    ]
}

# Start an MLflow run and log the model
with mlflow.start_run():
    mlflow.set_tags(tags_to_register)
    
    logged_agent_info = mlflow.pyfunc.log_model(
        name=model_name,
        python_model="agent.py",
        code_paths=["agent-config.yaml"],
        input_example=input_example,
        pip_requirements=[
            f"databricks-vectorsearch=={get_version('databricks-vectorsearch')}",
            f"databricks-langchain=={get_version('databricks-langchain')}",
            f"langchain=={get_version('langchain')}",
            f"mlflow=={get_version('mlflow')}",
        ],
        resources=resources
    )
    
    # Save the model URI for later use
    model_uri = logged_agent_info.model_uri
    
print(f"✅ Model logged successfully!")
print(f"Model URI: {model_uri}")

🔗 View Logged Model at: https://adb-1969779519092963.3.azuredatabricks.net/ml/experiments/3522241114100384/models/m-d4f79ccd2b81418da8d86de25b108be9?o=1969779519092963
Tool name dmi_genomics_qa_team_consenting_practice__bg_genai__docs_chunked_index is too long, truncating to 64 characters nomics_qa_team_consenting_practice__bg_genai__docs_chunked_index.
2026/03/10 11:28:02 INFO mlflow.pyfunc: Predicting on input example to validate output
2026/03/10 11:28:02 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Tool name dmi_genomics_qa_team_consenting_practice__bg_genai__docs_chunked_index is too long, truncating to 64 characters nomics_qa_team_consenting_practice__bg_genai__docs_chunked_index.
2026/03/10 11:28:15 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.


✅ Model logged successfully!
Model URI: models:/m-d4f79ccd2b81418da8d86de25b108be9


In [0]:
# Set the registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Define the fully qualified model name in Unity Catalog
UC_MODEL_NAME = "dmi_genomics_qa_team_consenting_practice.bg_genai.spark_knowledge_assistant"

# Register the model to Unity Catalog
uc_registered_model_info = mlflow.register_model(
    model_uri=model_uri, 
    name=UC_MODEL_NAME
)

print(f"✅ Model registered successfully to Unity Catalog!")
print(f"Model Name: {UC_MODEL_NAME}")
print(f"Version: {uc_registered_model_info.version}")

Successfully registered model 'dmi_genomics_qa_team_consenting_practice.bg_genai.spark_knowledge_assistant'.


Uploading artifacts:   0%|          | 0/14 [00:00<?, ?it/s]

✅ Model registered successfully to Unity Catalog!
Model Name: dmi_genomics_qa_team_consenting_practice.bg_genai.spark_knowledge_assistant
Version: 1


🔗 Created version '1' of model 'dmi_genomics_qa_team_consenting_practice.bg_genai.spark_knowledge_assistant': https://adb-1969779519092963.3.azuredatabricks.net/explore/data/models/dmi_genomics_qa_team_consenting_practice/bg_genai/spark_knowledge_assistant/version/1?o=1969779519092963


In [0]:
# Load the model from the model URI (pyfunc flavor)
pyfunc_model = mlflow.pyfunc.load_model(model_uri)

# Use the input example that was logged with the model
input_data = pyfunc_model.input_example

print("Input data:")
print(input_data)
print("\n" + "="*50 + "\n")

# Make a prediction using the loaded model
result = mlflow.models.predict(
    model_uri=model_uri,
    input_data=input_data,
    env_manager="uv",
)

print("Agent Response:")
print(result)

Tool name dmi_genomics_qa_team_consenting_practice__bg_genai__docs_chunked_index is too long, truncating to 64 characters nomics_qa_team_consenting_practice__bg_genai__docs_chunked_index.


Input data:
{'input': [{'role': 'user', 'content': 'What is Spark?'}]}




2026/03/10 11:48:41 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


2026/03/10 11:48:42 INFO mlflow.utils.virtualenv: Creating a new environment in /tmp/virtualenv_envs/mlflow-4fdb836c9f0197b22997af6f5346b73d214a5c68 with python version 3.12.3 using uv
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: /tmp/virtualenv_envs/mlflow-4fdb836c9f0197b22997af6f5346b73d214a5c68
Activate with: source /tmp/virtualenv_envs/mlflow-4fdb836c9f0197b22997af6f5346b73d214a5c68/bin/activate
2026/03/10 11:48:44 INFO mlflow.utils.virtualenv: Installing dependencies
Using Python 3.12.3 environment at: /tmp/virtualenv_envs/mlflow-4fdb836c9f0197b22997af6f5346b73d214a5c68
Resolved 3 packages in 47ms
Prepared 3 packages in 111ms
Installed 3 packages in 19ms
 + pip==25.0.1
 + setuptools==74.0.0
 + wheel==0.45.1
Using Python 3.12.3 environment at: /tmp/virtualenv_envs/mlflow-4fdb836c9f0197b22997af6f5346b73d214a5c68
Resolved 156 packages in 889ms
Prepared 155 packages in 3.36s
Installed 155 packages in 458ms
 + aiohappyeyeballs==2.6.1
 + aioh

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
{"object": "response", "output": [{"type": "message", "id": "c9e70fa7-db2b-4f91-aec9-b8188e278f09", "content": [{"text": "**Spark** is a distributed\u2011computing framework that coordinates the execution of data\u2011processing tasks across a cluster of machines. It provides a runtime that abstracts a group of computers as a single resource pool, allowing you to write programs that operate on large data sets far beyond what a single machine can handle.  \n\n- **Cluster coordination** \u2013 Spark works with a cluster manager (e.g., Spark\u2019s own Standalone manager, YARN, or Mesos) that allocates resources to each Spark application.  \n- **Spark applications** \u2013 Users submit an application (via `spark-submit`) written in one of Spark\u2019s supported languages (Scala, Jav

2026/03/10 11:49:07 INFO mlflow.tracing.export.async_export_queue: Flushing the async trace logging queue before program exit. This may take a while...


Agent Response:
None
